# Lyapunov Stability & Control Barrier Functions
## Safe Control via CLF-CBF Quadratic Programs

This notebook provides a complete, from-scratch treatment of **Lyapunov stability theory** and **Control Barrier Functions (CBFs)** — the mathematical foundations for provably safe robotic control. We build from stability analysis to CBF-CLF-QP controllers that guarantee both convergence and collision avoidance.

**What you'll learn:**
1. Lyapunov's direct method for stability analysis of nonlinear systems
2. Control Lyapunov Functions (CLFs) for stabilization of control-affine systems
3. Control Barrier Functions (CBFs) for forward invariance and safety guarantees
4. CLF-CBF Quadratic Program (QP) synthesis for safe controllers
5. Applications: safe 2D navigation and adaptive cruise control

**Prerequisites:** Dynamical systems, quadratic programming (convex optimization basics), control theory (LQR, PID).

**References:**
- Ames, A.D. et al. *Control Barrier Functions: Theory and Applications*, European Control Conference, 2019.
- Khalil, H.K. *Nonlinear Systems*, 3rd ed., Prentice-Hall, 2002.
- Ames, A.D. et al. *Control Barrier Function Based Quadratic Programs for Safety Critical Systems*, IEEE TAC, 2017.
- See also: [LQR notebook](../lqr-control/lqr_control.ipynb), [Convex Optimization notebook](../../maths/optimization/convex-optimization/convex_optimization.ipynb), [Potential Fields notebook](../potential-fields/potential_fields.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, linprog
from matplotlib.patches import Circle
from matplotlib.collections import LineCollection

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

# Simulation parameters
DT = 0.01               # Integration time step (s)
T_SIM = 15.0            # Default simulation time (s)

# Navigation parameters
X_GOAL = np.array([8.0, 8.0])    # Goal position for 2D navigation
V_MAX = 2.0                       # Maximum velocity for single integrator

# Adaptive cruise control parameters
V_DES = 30.0            # Desired cruise speed (m/s)
TAU_H = 1.8             # Time headway (s)
A_MAX = 3.0             # Max acceleration (m/s^2)
A_MIN = -5.0            # Max braking deceleration (m/s^2)

# CBF/CLF parameters
GAMMA_CLF = 1.0         # CLF convergence rate
ALPHA_CBF = 1.0         # CBF class-K function coefficient
P_SLACK = 100.0         # Penalty on CLF relaxation slack variable

# Numerical tolerance
TOL = 1e-8

# Plot colors
C_BLUE = 'steelblue'
C_RED = 'coral'
C_GREEN = 'seagreen'
C_GOLD = 'goldenrod'
C_PURPLE = 'mediumpurple'
C_SAFE = '#2ecc71'
C_UNSAFE = '#e74c3c'

---
## 1. Introduction — Stability and Safety

Autonomous systems must do more than reach a goal — they must **provably avoid collisions** along the way. Classical controllers like PID or LQR provide no formal safety guarantees. Modern safety-critical control bridges this gap using **Lyapunov stability** (for convergence) and **Control Barrier Functions** (for constraint satisfaction).

| Method | Stability | Safety | Optimality | Constraints |
|--------|-----------|--------|------------|-------------|
| PID | No formal guarantee | None | No | No |
| LQR | Asymptotically stable | None | Quadratic cost | No (unconstrained) |
| CLF | Lyapunov-certified | None | No | No |
| CBF | None | Barrier-certified | No | Safety only |
| **CLF-CBF-QP** | **Lyapunov-certified** | **Barrier-certified** | **Pointwise min-norm** | **Safety + input bounds** |

**Key insight:** By combining CLF and CBF constraints into a single **Quadratic Program (QP)** solved at each timestep, we obtain a controller that is both stabilizing and safe — with safety taking strict priority over convergence through a relaxation slack variable.

**Roadmap:**
1. Lyapunov stability theory → stability without control
2. Control Lyapunov Functions → stability with control
3. Control Barrier Functions → safety with control
4. CLF-CBF-QP → stability + safety
5. Applications → 2D navigation, adaptive cruise control
6. Higher-relative-degree CBFs → handling underactuated constraints

---
## 2. Lyapunov Stability Theory

### Equilibrium and Stability

Consider an autonomous system $\dot{x} = f(x)$ where $x \in \mathbb{R}^n$. A point $x_e$ is an **equilibrium** if $f(x_e) = 0$.

**Stability definitions** (Lyapunov sense):
- **Stable:** trajectories starting near $x_e$ stay near $x_e$
- **Asymptotically stable:** stable + trajectories converge to $x_e$
- **Exponentially stable:** converges at rate $\|x(t)\| \leq k e^{-\lambda t} \|x(0)\|$

### Lyapunov's Direct Method

Instead of solving the ODE, we analyze stability using an **energy-like** function.

**Theorem (Lyapunov).** Let $V : \mathbb{R}^n \to \mathbb{R}$ be continuously differentiable with $V(0) = 0$. If:
1. $V(x) > 0$ for all $x \neq 0$ (positive definite)
2. $\dot{V}(x) = \nabla V(x) \cdot f(x) \leq 0$ for all $x$ (negative semi-definite)

then the origin is **stable**. If additionally $\dot{V}(x) < 0$ for all $x \neq 0$, the origin is **asymptotically stable**.

### Class $\mathcal{K}$ and $\mathcal{K}_\infty$ Functions

A continuous function $\alpha : [0, \infty) \to [0, \infty)$ is **class $\mathcal{K}$** if $\alpha(0) = 0$ and it is strictly increasing. It is **class $\mathcal{K}_\infty$** if additionally $\alpha(r) \to \infty$ as $r \to \infty$.

Common choices: $\alpha(r) = \gamma r$ (linear), $\alpha(r) = r^2$ (quadratic).

These functions are used to bound Lyapunov decay rates: $\dot{V}(x) \leq -\alpha(V(x))$.

In [ ]:
def rk4_step(f, x, u, dt):
    """Runge-Kutta 4th-order integration step.

    Args:
        f: Dynamics function f(x, u) -> dx/dt. Shape: (n,) -> (n,).
        x: Current state. Shape: (n,).
        u: Control input. Shape: (m,).
        dt: Time step (scalar).

    Returns:
        x_next: Next state. Shape: (n,).
    """
    k1 = f(x, u)
    k2 = f(x + 0.5 * dt * k1, u)
    k3 = f(x + 0.5 * dt * k2, u)
    k4 = f(x + dt * k3, u)
    return x + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def simulate(dynamics, controller, x0, T, dt=DT):
    """Simulate a controlled dynamical system.

    Args:
        dynamics: Function dynamics(x, u) -> dx/dt. Shape: (n,) -> (n,).
        controller: Function controller(x, t) -> u. Shape: (n,) -> (m,).
        x0: Initial state. Shape: (n,).
        T: Simulation duration (scalar).
        dt: Time step (scalar).

    Returns:
        t_hist: Time history. Shape: (N,).
        x_hist: State history. Shape: (N, n).
        u_hist: Control history. Shape: (N-1, m).
    """
    N = int(T / dt)
    n = x0.shape[0]
    x_hist = np.zeros((N + 1, n))
    x_hist[0] = x0
    u_hist = []
    t_hist = np.linspace(0, T, N + 1)

    for i in range(N):
        u = controller(x_hist[i], t_hist[i])
        u_hist.append(u)
        x_hist[i + 1] = rk4_step(dynamics, x_hist[i], u, dt)

    u_hist = np.array(u_hist)
    return t_hist, x_hist, u_hist


# ---- Teaching Example: Lyapunov analysis ----
# System: dx1/dt = -x1 + x2^2, dx2/dt = -x2
# Lyapunov candidate: V(x) = x1^2 + x2^2

def lyapunov_example_dynamics(x, u=None):
    """Autonomous system: dx1 = -x1 + x2^2, dx2 = -x2.

    Args:
        x: State vector. Shape: (2,).
        u: Unused control input.

    Returns:
        dx: State derivative. Shape: (2,).
    """
    return np.array([-x[0] + x[1]**2, -x[1]])


def lyapunov_V(x):
    """Lyapunov function V = x1^2 + x2^2.

    Args:
        x: State vector. Shape: (2,).

    Returns:
        V: Lyapunov function value (scalar).
    """
    return x[0]**2 + x[1]**2


def lyapunov_Vdot(x):
    """Time derivative of V along system trajectories.

    dV/dt = 2*x1*(-x1+x2^2) + 2*x2*(-x2) = -2*x1^2 + 2*x1*x2^2 - 2*x2^2

    Args:
        x: State vector. Shape: (2,).

    Returns:
        Vdot: Time derivative of V (scalar).
    """
    f = lyapunov_example_dynamics(x)
    grad_V = np.array([2*x[0], 2*x[1]])
    return grad_V @ f


# Verify Vdot <= 0 on a grid (for |x| small enough)
grid = np.linspace(-1.0, 1.0, 50)
X1, X2 = np.meshgrid(grid, grid)
Vdot_grid = np.zeros_like(X1)
for i in range(X1.shape[0]):
    for j in range(X1.shape[1]):
        Vdot_grid[i, j] = lyapunov_Vdot(np.array([X1[i, j], X2[i, j]]))

# Check: on this grid, is Vdot <= 0 everywhere?
max_Vdot = np.max(Vdot_grid)
# Near origin where |x2^2| < |x1|, Vdot should be negative
# For larger x2, Vdot can be positive — check a smaller region
grid_small = np.linspace(-0.5, 0.5, 50)
X1s, X2s = np.meshgrid(grid_small, grid_small)
Vdot_small = np.zeros_like(X1s)
for i in range(X1s.shape[0]):
    for j in range(X1s.shape[1]):
        Vdot_small[i, j] = lyapunov_Vdot(np.array([X1s[i, j], X2s[i, j]]))

max_Vdot_small = np.max(Vdot_small)
status = "PASS" if max_Vdot_small <= TOL else "FAIL"
print(f"Vdot <= 0 on [-0.5, 0.5]^2 grid (max Vdot = {max_Vdot_small:.6f}): [{status}]")

# Simulate from multiple initial conditions
ics = [np.array([0.8, 0.4]), np.array([-0.5, 0.3]),
       np.array([0.3, -0.6]), np.array([-0.4, -0.5])]
trajectories = []
for ic in ics:
    t, x, _ = simulate(lyapunov_example_dynamics, lambda x, t: np.zeros(1), ic, 10.0)
    trajectories.append((t, x))

# Verify convergence
final_norms = [np.linalg.norm(traj[1][-1]) for traj in trajectories]
status = "PASS" if all(n < 0.01 for n in final_norms) else "FAIL"
print(f"All trajectories converge to origin (max final norm = {max(final_norms):.6f}): [{status}]")

The system $\dot{x}_1 = -x_1 + x_2^2$, $\dot{x}_2 = -x_2$ has the origin as an equilibrium. Using $V = x_1^2 + x_2^2$, we showed $\dot{V} \leq 0$ in a neighborhood of the origin, confirming local stability. The phase portrait below shows level sets of $V$, coloring by $\dot{V}$, and trajectories converging to the origin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Phase portrait with V level sets and Vdot coloring
ax = axes[0]
grid_plot = np.linspace(-1.0, 1.0, 40)
X1p, X2p = np.meshgrid(grid_plot, grid_plot)
Vdot_plot = np.zeros_like(X1p)
U_field = np.zeros_like(X1p)
V_field = np.zeros_like(X1p)
V_vals = np.zeros_like(X1p)

for i in range(X1p.shape[0]):
    for j in range(X1p.shape[1]):
        state = np.array([X1p[i, j], X2p[i, j]])
        f = lyapunov_example_dynamics(state)
        U_field[i, j] = f[0]
        V_field[i, j] = f[1]
        Vdot_plot[i, j] = lyapunov_Vdot(state)
        V_vals[i, j] = lyapunov_V(state)

# Color background by Vdot
cf = ax.contourf(X1p, X2p, Vdot_plot, levels=50, cmap='RdYlGn_r', alpha=0.6)
plt.colorbar(cf, ax=ax, label=r'$\dot{V}(x)$')

# V level sets
ax.contour(X1p, X2p, V_vals, levels=[0.1, 0.3, 0.5, 0.8, 1.0],
           colors='gray', linestyles='--', linewidths=0.8)

# Vector field
ax.quiver(X1p[::3, ::3], X2p[::3, ::3], U_field[::3, ::3], V_field[::3, ::3],
          color='black', alpha=0.4, scale=15)

# Trajectories
colors_traj = [C_BLUE, C_RED, C_GREEN, C_PURPLE]
for idx, (t, x) in enumerate(trajectories):
    ax.plot(x[:, 0], x[:, 1], color=colors_traj[idx], linewidth=2, alpha=0.9)
    ax.plot(x[0, 0], x[0, 1], 'o', color=colors_traj[idx], markersize=8)

ax.plot(0, 0, 'k*', markersize=15, label='Equilibrium')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('Phase Portrait with Lyapunov Analysis')
ax.legend(loc='upper left')
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-1.0, 1.0)

# Right: V(t) decay for each trajectory
ax = axes[1]
for idx, (t, x) in enumerate(trajectories):
    V_t = np.array([lyapunov_V(x[i]) for i in range(len(t))])
    ax.plot(t, V_t, color=colors_traj[idx], linewidth=2,
            label=f'IC=({ics[idx][0]:.1f}, {ics[idx][1]:.1f})')

ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$V(x(t))$')
ax.set_title(r'Lyapunov Function Decay $V(t) \to 0$')
ax.legend()
ax.set_ylim(bottom=-0.05)

plt.tight_layout()
plt.show()

---
## 3. Control Lyapunov Functions (CLFs)

### Control-Affine Systems

Many robotic systems have the form:

$$\dot{x} = f(x) + g(x) u$$

where $f(x)$ is the drift dynamics, $g(x)$ is the input matrix, and $u \in \mathbb{R}^m$ is the control input.

### Lie Derivatives

The time derivative of a Lyapunov candidate $V(x)$ along the controlled system is:

$$\dot{V} = \nabla V \cdot (f(x) + g(x)u) = \underbrace{\nabla V \cdot f(x)}_{L_f V} + \underbrace{\nabla V \cdot g(x)}_{L_g V} \cdot u$$

where $L_f V$ and $L_g V$ are the **Lie derivatives** of $V$ along $f$ and $g$ respectively.

### CLF Condition

A positive definite function $V(x)$ is a **Control Lyapunov Function** if:

$$\inf_{u} \left[ L_f V(x) + L_g V(x) \cdot u \right] \leq -\gamma V(x) \quad \forall x \neq 0$$

This ensures there always exists a control $u$ that makes $V$ decrease at rate $\gamma$.

### Sontag's Universal Formula

When $L_g V \neq 0$, Sontag's formula gives a closed-form stabilizing controller:

$$u = \begin{cases} -\frac{L_f V + \sqrt{(L_f V)^2 + (L_g V \cdot L_g V^\top)^2}}{L_g V \cdot L_g V^\top} L_g V^\top & \text{if } L_g V \neq 0 \\ 0 & \text{if } L_g V = 0 \end{cases}$$

This controller is smooth (away from the origin) and achieves $\dot{V} < 0$.

In [ ]:
def clf_sontag(x, f_x, g_x, V_val, LfV, LgV, gamma=GAMMA_CLF):
    """Sontag's universal formula for CLF-based control.

    Args:
        x: State vector. Shape: (n,).
        f_x: Drift dynamics f(x). Shape: (n,).
        g_x: Input matrix g(x). Shape: (n, m).
        V_val: Lyapunov function value (scalar).
        LfV: Lie derivative L_f V (scalar).
        LgV: Lie derivative L_g V. Shape: (m,).
        gamma: CLF convergence rate (scalar).

    Returns:
        u: Control input. Shape: (m,).
    """
    LgV = np.atleast_1d(LgV)
    m = LgV.shape[0]
    LgV_norm_sq = LgV @ LgV

    if LgV_norm_sq < TOL:
        return np.zeros(m)

    a = LfV + gamma * V_val
    b_sq = LgV_norm_sq

    # Sontag's formula: u = -(a + sqrt(a^2 + b^2)) / b * LgV  (when a > 0)
    # More precisely: ensures Vdot <= -gamma*V
    if a <= 0:
        # Already satisfies CLF condition without control
        return np.zeros(m)

    u = -(a + np.sqrt(a**2 + b_sq**2)) / b_sq * LgV
    return u


# ---- Teaching example: 2D control-affine system ----
# System: dx = [x2; -x1 + (1-x1^2)*x2] + [0; 1]*u  (Van der Pol-like)
# CLF: V = x1^2 + x2^2

def clf_example_f(x):
    """Drift dynamics for CLF example.

    Args:
        x: State vector. Shape: (2,).

    Returns:
        f: Drift. Shape: (2,).
    """
    return np.array([x[1], -x[0] + (1 - x[0]**2) * x[1]])

def clf_example_g(x):
    """Input matrix for CLF example.

    Args:
        x: State vector. Shape: (2,).

    Returns:
        g: Input matrix. Shape: (2, 1).
    """
    return np.array([[0.0], [1.0]])

def clf_example_dynamics(x, u):
    """Full dynamics dx = f(x) + g(x)*u.

    Args:
        x: State vector. Shape: (2,).
        u: Control input. Shape: (1,).

    Returns:
        dx: State derivative. Shape: (2,).
    """
    return clf_example_f(x) + clf_example_g(x).flatten() * u[0]

def clf_controller(x, t):
    """CLF controller using Sontag's formula.

    Args:
        x: State vector. Shape: (2,).
        t: Time (scalar, unused).

    Returns:
        u: Control input. Shape: (1,).
    """
    V_val = x[0]**2 + x[1]**2
    grad_V = np.array([2*x[0], 2*x[1]])
    f_x = clf_example_f(x)
    g_x = clf_example_g(x)
    LfV = grad_V @ f_x
    LgV = grad_V @ g_x  # Shape: (1,)
    u = clf_sontag(x, f_x, g_x, V_val, LfV, LgV.flatten(), gamma=GAMMA_CLF)
    return u

# Simulate CLF-stabilized trajectories
clf_ics = [np.array([2.0, 1.0]), np.array([-1.5, 2.0]),
           np.array([1.0, -2.0]), np.array([-2.0, -1.0])]

clf_trajs = []
for ic in clf_ics:
    t, x, u = simulate(clf_example_dynamics, clf_controller, ic, 8.0)
    clf_trajs.append((t, x, u))

# Open-loop (no control) trajectories for comparison
openloop_trajs = []
for ic in clf_ics:
    t, x, _ = simulate(clf_example_dynamics, lambda x, t: np.zeros(1), ic, 8.0)
    openloop_trajs.append((t, x))

# Verify CLF stabilization
clf_final_norms = [np.linalg.norm(traj[1][-1]) for traj in clf_trajs]
status = "PASS" if all(n < 0.05 for n in clf_final_norms) else "FAIL"
print(f"CLF stabilizes all trajectories (max final norm = {max(clf_final_norms):.6f}): [{status}]")

# Verify open-loop diverges (Van der Pol has limit cycle)
ol_final_norms = [np.linalg.norm(traj[1][-1]) for traj in openloop_trajs]
status = "PASS" if any(n > 0.5 for n in ol_final_norms) else "FAIL"
print(f"Open-loop does NOT converge (max final norm = {max(ol_final_norms):.4f}): [{status}]")

The CLF controller (Sontag's formula) successfully stabilizes the Van der Pol-like oscillator, which otherwise exhibits a limit cycle in open loop. Below we compare CLF-controlled vs. open-loop phase portraits and show the Lyapunov function decay over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Phase portrait comparison
ax = axes[0]
for idx in range(len(clf_ics)):
    # Open-loop (dashed)
    t_ol, x_ol = openloop_trajs[idx]
    ax.plot(x_ol[:, 0], x_ol[:, 1], '--', color=colors_traj[idx], alpha=0.4, linewidth=1.5)
    # CLF-controlled (solid)
    t_cl, x_cl, _ = clf_trajs[idx]
    ax.plot(x_cl[:, 0], x_cl[:, 1], '-', color=colors_traj[idx], linewidth=2,
            label=f'CLF IC=({clf_ics[idx][0]:.1f}, {clf_ics[idx][1]:.1f})')
    ax.plot(clf_ics[idx][0], clf_ics[idx][1], 'o', color=colors_traj[idx], markersize=8)

ax.plot(0, 0, 'k*', markersize=15, label='Origin')
# V level sets
theta = np.linspace(0, 2*np.pi, 100)
for r in [0.5, 1.0, 1.5, 2.0]:
    ax.plot(r*np.cos(theta), r*np.sin(theta), 'gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('Phase Portrait: CLF (solid) vs Open-Loop (dashed)')
ax.legend(fontsize=9, loc='upper left')
ax.set_aspect('equal')

# Right: V(t) decay
ax = axes[1]
for idx, (t, x, u) in enumerate(clf_trajs):
    V_t = np.array([x[i, 0]**2 + x[i, 1]**2 for i in range(len(t))])
    ax.plot(t, V_t, color=colors_traj[idx], linewidth=2,
            label=f'IC=({clf_ics[idx][0]:.1f}, {clf_ics[idx][1]:.1f})')

ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$V(x(t))$')
ax.set_title(r'CLF Lyapunov Decay $V(t) \to 0$')
ax.legend()
ax.set_ylim(bottom=-0.1)

plt.tight_layout()
plt.show()

---
## 4. Control Barrier Functions (CBFs)

### Safe Set and Forward Invariance

Define a **safe set** via a continuously differentiable function $h : \mathbb{R}^n \to \mathbb{R}$:

$$\mathcal{C} = \{x \in \mathbb{R}^n : h(x) \geq 0\}$$

The set $\mathcal{C}$ is **forward invariant** if $x(0) \in \mathcal{C} \implies x(t) \in \mathcal{C}$ for all $t \geq 0$.

### CBF Condition

For a control-affine system $\dot{x} = f(x) + g(x)u$, the function $h$ is a **Control Barrier Function** if there exists an extended class $\mathcal{K}_\infty$ function $\alpha$ such that:

$$\sup_{u} \left[ L_f h(x) + L_g h(x) \cdot u \right] \geq -\alpha(h(x)) \quad \forall x \in \mathcal{C}$$

**Intuition:** The CBF condition ensures we can always find a control that keeps $h$ from decreasing too fast — preventing the system from reaching the boundary $h = 0$ and exiting the safe set.

Any control $u$ satisfying:

$$L_f h(x) + L_g h(x) \cdot u \geq -\alpha(h(x))$$

renders $\mathcal{C}$ forward invariant. This is a **linear constraint** in $u$ — perfect for QP formulation.

In [ ]:
# ---- Teaching Example: Single integrator with circular obstacle ----
# System: dx = u (single integrator in 2D)
# Obstacle at x_obs with radius r
# CBF: h(x) = ||x - x_obs||^2 - r^2

OBS_CENTER = np.array([4.0, 3.0])
OBS_RADIUS = 1.5
CBF_START = np.array([1.0, 1.0])
CBF_TARGET = np.array([7.0, 5.0])

def cbf_h(x, obs_c=OBS_CENTER, obs_r=OBS_RADIUS):
    """CBF for circular obstacle avoidance: h = ||x - x_obs||^2 - r^2.

    Args:
        x: Position. Shape: (2,).
        obs_c: Obstacle center. Shape: (2,).
        obs_r: Obstacle radius (scalar).

    Returns:
        h: Barrier function value (scalar). h >= 0 means safe.
    """
    return np.sum((x - obs_c)**2) - obs_r**2

def cbf_grad_h(x, obs_c=OBS_CENTER):
    """Gradient of CBF h w.r.t. x.

    Args:
        x: Position. Shape: (2,).
        obs_c: Obstacle center. Shape: (2,).

    Returns:
        grad_h: Gradient. Shape: (2,).
    """
    return 2.0 * (x - obs_c)

def naive_controller(x, t, target=CBF_TARGET):
    """Go-to-goal controller (no obstacle avoidance).

    Args:
        x: Position. Shape: (2,).
        t: Time (unused).
        target: Goal position. Shape: (2,).

    Returns:
        u: Velocity command. Shape: (2,).
    """
    direction = target - x
    norm = np.linalg.norm(direction)
    if norm < TOL:
        return np.zeros(2)
    return V_MAX * direction / norm

def cbf_safe_controller(x, t, target=CBF_TARGET, alpha=ALPHA_CBF):
    """CBF-constrained go-to-goal controller via QP.

    Args:
        x: Position. Shape: (2,).
        t: Time (unused).
        target: Goal position. Shape: (2,).
        alpha: Class-K coefficient (scalar).

    Returns:
        u: Safe velocity command. Shape: (2,).
    """
    # Nominal: go to goal
    u_nom = naive_controller(x, t, target)

    # Single integrator: f(x)=0, g(x)=I  =>  Lfh=0, Lgh=grad_h
    h_val = cbf_h(x)
    grad_h = cbf_grad_h(x)

    # CBF constraint: grad_h @ u >= -alpha * h
    # min ||u - u_nom||^2 s.t. grad_h @ u >= -alpha * h, ||u|| <= V_MAX
    def objective(u):
        return np.sum((u - u_nom)**2)

    def cbf_constraint(u):
        return grad_h @ u + alpha * h_val  # >= 0

    def speed_constraint(u):
        return V_MAX**2 - np.sum(u**2)  # >= 0

    constraints = [
        {'type': 'ineq', 'fun': cbf_constraint},
        {'type': 'ineq', 'fun': speed_constraint},
    ]

    result = minimize(objective, u_nom, method='SLSQP', constraints=constraints)
    return result.x

# Simulate both controllers
si_dynamics = lambda x, u: u  # single integrator

t_naive, x_naive, _ = simulate(si_dynamics, naive_controller, CBF_START, 5.0)
t_safe, x_safe, _ = simulate(si_dynamics, cbf_safe_controller, CBF_START, 6.0)

# Verify: naive hits obstacle, safe avoids it
min_dist_naive = np.min(np.linalg.norm(x_naive - OBS_CENTER, axis=1))
min_dist_safe = np.min(np.linalg.norm(x_safe - OBS_CENTER, axis=1))

status = "PASS" if min_dist_naive < OBS_RADIUS else "FAIL"
print(f"Naive controller violates safety (min dist = {min_dist_naive:.4f} < r = {OBS_RADIUS}): [{status}]")

status = "PASS" if min_dist_safe >= OBS_RADIUS - 0.05 else "FAIL"
print(f"CBF controller maintains safety (min dist = {min_dist_safe:.4f} >= r = {OBS_RADIUS}): [{status}]")

# Verify h >= 0 along safe trajectory
h_safe_traj = np.array([cbf_h(x_safe[i]) for i in range(len(t_safe))])
min_h = np.min(h_safe_traj)
status = "PASS" if min_h >= -0.1 else "FAIL"
print(f"CBF h(x) >= 0 along safe trajectory (min h = {min_h:.6f}): [{status}]")

The naive go-to-goal controller drives straight through the obstacle, while the CBF-constrained controller safely navigates around it. The safe set boundary ($h = 0$) is the circle around the obstacle — the CBF ensures the trajectory never enters this region.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Shade unsafe region
theta = np.linspace(0, 2*np.pi, 200)
obs_x = OBS_CENTER[0] + OBS_RADIUS * np.cos(theta)
obs_y = OBS_CENTER[1] + OBS_RADIUS * np.sin(theta)
ax.fill(obs_x, obs_y, color=C_UNSAFE, alpha=0.3, label='Unsafe region ($h < 0$)')
ax.plot(obs_x, obs_y, color=C_UNSAFE, linewidth=2, linestyle='-')

# Color safe region lightly
grid_bg = np.linspace(-1, 9, 100)
Xbg, Ybg = np.meshgrid(grid_bg, grid_bg)
H_bg = np.zeros_like(Xbg)
for i in range(Xbg.shape[0]):
    for j in range(Xbg.shape[1]):
        H_bg[i, j] = cbf_h(np.array([Xbg[i, j], Ybg[i, j]]))
ax.contourf(Xbg, Ybg, H_bg, levels=[0, 100], colors=[C_SAFE], alpha=0.08)

# Trajectories
ax.plot(x_naive[:, 0], x_naive[:, 1], '--', color=C_RED, linewidth=2.5,
        label='Naive (unsafe)', alpha=0.8)
ax.plot(x_safe[:, 0], x_safe[:, 1], '-', color=C_BLUE, linewidth=2.5,
        label='CBF-safe')

# Start and goal
ax.plot(CBF_START[0], CBF_START[1], 'go', markersize=12, label='Start', zorder=5)
ax.plot(CBF_TARGET[0], CBF_TARGET[1], 'r*', markersize=15, label='Goal', zorder=5)
ax.plot(OBS_CENTER[0], OBS_CENTER[1], 'kx', markersize=10, markeredgewidth=2)

ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('CBF Obstacle Avoidance: Safe vs. Naive Controller')
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-0.5, 6.5)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 5. CLF-CBF Quadratic Program (QP) Controller

### Unifying Stability and Safety

The key insight of modern safety-critical control is combining CLF and CBF constraints into a single **Quadratic Program** solved at each timestep.

### The CLF-CBF-QP Formulation

$$\min_{u, \delta} \quad \|u - u_{\text{ref}}\|^2 + p \, \delta^2$$

subject to:

$$L_f V(x) + L_g V(x) \cdot u \leq -\gamma \, V(x) + \delta \qquad \text{(CLF constraint, relaxed)}$$

$$L_f h_i(x) + L_g h_i(x) \cdot u \geq -\alpha_i(h_i(x)) \qquad \text{(CBF constraints, hard)}$$

$$u_{\min} \leq u \leq u_{\max} \qquad \text{(input bounds)}$$

**Key design choices:**
- The **slack variable** $\delta$ relaxes the CLF constraint — this ensures feasibility when safety and stability conflict. Safety is **never** relaxed.
- The penalty $p$ controls the trade-off: large $p$ prioritizes stability, but safety always wins.
- $u_{\text{ref}}$ is a nominal/desired control (e.g., from a planner).
- Multiple CBF constraints can handle multiple obstacles simultaneously.

In [ ]:
def clf_cbf_qp(x, f_x, g_x, V_val, grad_V, h_vals, grad_h_list,
               u_ref, u_dim=2, u_min=None, u_max=None,
               gamma=GAMMA_CLF, alpha=ALPHA_CBF, p_slack=P_SLACK):
    """Solve the CLF-CBF-QP for safe stabilizing control.

    Minimizes ||u - u_ref||^2 + p * delta^2
    subject to:
        L_f V + L_g V * u <= -gamma * V + delta   (CLF, relaxed)
        L_f h_i + L_g h_i * u >= -alpha * h_i     (CBF, hard)
        u_min <= u <= u_max                         (input bounds)

    Args:
        x: State vector. Shape: (n,).
        f_x: Drift dynamics f(x). Shape: (n,).
        g_x: Input matrix g(x). Shape: (n, m).
        V_val: CLF value (scalar).
        grad_V: Gradient of V. Shape: (n,).
        h_vals: List of CBF values. Length: num_cbf.
        grad_h_list: List of CBF gradients. Each shape: (n,).
        u_ref: Reference control. Shape: (m,).
        u_dim: Control dimension (int).
        u_min: Lower input bounds. Shape: (m,) or None.
        u_max: Upper input bounds. Shape: (m,) or None.
        gamma: CLF convergence rate (scalar).
        alpha: CBF class-K coefficient (scalar).
        p_slack: Slack penalty weight (scalar).

    Returns:
        u_opt: Optimal control. Shape: (m,).
        delta: CLF slack value (scalar).
        success: Whether QP was solved successfully (bool).
    """
    m = u_dim
    # Decision variable: z = [u (m), delta (1)]
    n_var = m + 1

    # Lie derivatives
    LfV = grad_V @ f_x
    LgV = grad_V @ g_x  # Shape: (m,) or (1, m)
    if LgV.ndim > 1:
        LgV = LgV.flatten()

    def objective(z):
        u = z[:m]
        delta = z[m]
        return np.sum((u - u_ref)**2) + p_slack * delta**2

    def objective_jac(z):
        u = z[:m]
        delta = z[m]
        grad = np.zeros(n_var)
        grad[:m] = 2.0 * (u - u_ref)
        grad[m] = 2.0 * p_slack * delta
        return grad

    constraints = []

    # CLF constraint: LfV + LgV @ u <= -gamma * V + delta
    # Rewrite: -(LfV + LgV @ u + gamma * V - delta) >= 0
    def clf_con(z):
        u = z[:m]
        delta = z[m]
        return -(LfV + LgV @ u + gamma * V_val - delta)

    constraints.append({'type': 'ineq', 'fun': clf_con})

    # CBF constraints: Lfh_i + Lgh_i @ u >= -alpha * h_i
    for k in range(len(h_vals)):
        grad_h_k = grad_h_list[k]
        h_k = h_vals[k]
        Lfh = grad_h_k @ f_x
        Lgh = grad_h_k @ g_x
        if Lgh.ndim > 1:
            Lgh = Lgh.flatten()

        def cbf_con(z, Lfh=Lfh, Lgh=Lgh, h_k=h_k):
            u = z[:m]
            return Lfh + Lgh @ u + alpha * h_k

        constraints.append({'type': 'ineq', 'fun': cbf_con})

    # Input bounds
    bounds = []
    for i in range(m):
        lb = u_min[i] if u_min is not None else -1e6
        ub = u_max[i] if u_max is not None else 1e6
        bounds.append((lb, ub))
    bounds.append((None, None))  # delta is unbounded

    z0 = np.zeros(n_var)
    z0[:m] = u_ref

    result = minimize(objective, z0, jac=objective_jac, method='SLSQP',
                      constraints=constraints, bounds=bounds,
                      options={'ftol': 1e-10, 'maxiter': 200})

    u_opt = result.x[:m]
    delta = result.x[m]
    return u_opt, delta, result.success


def simulate_clf_cbf(dynamics_f, dynamics_g, V_func, grad_V_func,
                     h_funcs, grad_h_funcs, x0, T, dt=DT,
                     u_ref_func=None, u_dim=2, u_min=None, u_max=None,
                     gamma=GAMMA_CLF, alpha=ALPHA_CBF, p_slack=P_SLACK):
    """Simulate system with CLF-CBF-QP controller.

    Args:
        dynamics_f: Drift function f(x). Shape: (n,) -> (n,).
        dynamics_g: Input matrix function g(x). Shape: (n,) -> (n, m).
        V_func: CLF function V(x) -> scalar.
        grad_V_func: CLF gradient function. (n,) -> (n,).
        h_funcs: List of CBF functions h_i(x) -> scalar.
        grad_h_funcs: List of CBF gradient functions. (n,) -> (n,).
        x0: Initial state. Shape: (n,).
        T: Simulation time (scalar).
        dt: Time step (scalar).
        u_ref_func: Reference control function u_ref(x) -> (m,). Default: zeros.
        u_dim: Control dimension (int).
        u_min: Lower input bounds. Shape: (m,) or None.
        u_max: Upper input bounds. Shape: (m,) or None.
        gamma: CLF rate (scalar).
        alpha: CBF rate (scalar).
        p_slack: Slack penalty (scalar).

    Returns:
        t_hist: Time history. Shape: (N+1,).
        x_hist: State history. Shape: (N+1, n).
        u_hist: Control history. Shape: (N, m).
        delta_hist: Slack history. Shape: (N,).
    """
    N = int(T / dt)
    n = x0.shape[0]
    x_hist = np.zeros((N + 1, n))
    x_hist[0] = x0
    u_hist = []
    delta_hist = []
    t_hist = np.linspace(0, T, N + 1)

    for i in range(N):
        x = x_hist[i]
        f_x = dynamics_f(x)
        g_x = dynamics_g(x)
        V_val = V_func(x)
        gV = grad_V_func(x)
        h_vals = [h(x) for h in h_funcs]
        grad_hs = [gh(x) for gh in grad_h_funcs]

        if u_ref_func is not None:
            u_ref = u_ref_func(x)
        else:
            u_ref = np.zeros(u_dim)

        u_opt, delta, _ = clf_cbf_qp(
            x, f_x, g_x, V_val, gV, h_vals, grad_hs,
            u_ref, u_dim=u_dim, u_min=u_min, u_max=u_max,
            gamma=gamma, alpha=alpha, p_slack=p_slack
        )

        u_hist.append(u_opt)
        delta_hist.append(delta)

        dynamics = lambda x, u: dynamics_f(x) + dynamics_g(x) @ u
        x_hist[i + 1] = rk4_step(dynamics, x, u_opt, dt)

    return t_hist, x_hist, np.array(u_hist), np.array(delta_hist)

print("CLF-CBF-QP solver and simulation loop defined.")

### Visualizing the QP Feasible Region

At any given state $x$, the CLF and CBF constraints define **half-planes** in control space $u \in \mathbb{R}^2$. The feasible region is their intersection (with input bounds). The QP finds the closest point to $u_{\text{ref}}$ within this region.

In [ ]:
# Visualize QP feasible region at a specific state near the obstacle
x_test = np.array([3.0, 2.0])  # Near obstacle
f_test = np.zeros(2)  # Single integrator: f = 0
g_test = np.eye(2)    # g = I

# CLF: V = ||x - goal||^2
V_test = np.sum((x_test - CBF_TARGET)**2)
grad_V_test = 2.0 * (x_test - CBF_TARGET)

# CBF: h = ||x - obs||^2 - r^2
h_test = cbf_h(x_test)
grad_h_test = cbf_grad_h(x_test)

# Lie derivatives (single integrator: Lf=0, Lg=grad)
LfV_test = 0.0
LgV_test = grad_V_test
Lfh_test = 0.0
Lgh_test = grad_h_test

# Solve QP
u_ref_test = V_MAX * (CBF_TARGET - x_test) / np.linalg.norm(CBF_TARGET - x_test)
u_opt_test, delta_test, _ = clf_cbf_qp(
    x_test, f_test, g_test, V_test, grad_V_test,
    [h_test], [grad_h_test], u_ref_test, u_dim=2,
    u_min=np.array([-V_MAX, -V_MAX]), u_max=np.array([V_MAX, V_MAX])
)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Grid in control space
u1_range = np.linspace(-3, 3, 300)
u2_range = np.linspace(-3, 3, 300)
U1, U2 = np.meshgrid(u1_range, u2_range)

# CLF constraint region: LgV @ u <= -gamma*V (without slack for visualization)
CLF_region = LgV_test[0] * U1 + LgV_test[1] * U2 + GAMMA_CLF * V_test
ax.contourf(U1, U2, CLF_region, levels=[-1e6, 0], colors=[C_BLUE], alpha=0.15)
ax.contour(U1, U2, CLF_region, levels=[0], colors=[C_BLUE], linewidths=2)

# CBF constraint region: Lgh @ u >= -alpha*h
CBF_region = Lgh_test[0] * U1 + Lgh_test[1] * U2 + ALPHA_CBF * h_test
ax.contourf(U1, U2, CBF_region, levels=[0, 1e6], colors=[C_GREEN], alpha=0.15)
ax.contour(U1, U2, CBF_region, levels=[0], colors=[C_GREEN], linewidths=2)

# Input bounds
rect = plt.Rectangle((-V_MAX, -V_MAX), 2*V_MAX, 2*V_MAX,
                      fill=False, edgecolor=C_GOLD, linewidth=2, linestyle='--')
ax.add_patch(rect)

# Mark points
ax.plot(u_ref_test[0], u_ref_test[1], 's', color=C_RED, markersize=12,
        label=r'$u_{\rm ref}$ (nominal)', zorder=5)
ax.plot(u_opt_test[0], u_opt_test[1], '*', color='black', markersize=18,
        label=r'$u^*$ (QP solution)', zorder=5)

ax.set_xlabel(r'$u_1$')
ax.set_ylabel(r'$u_2$')
ax.set_title(f'QP Feasible Region at x = ({x_test[0]}, {x_test[1]})')

# Custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=C_BLUE, alpha=0.3, label='CLF feasible'),
    Patch(facecolor=C_GREEN, alpha=0.3, label='CBF feasible'),
    plt.Line2D([0], [0], color=C_GOLD, linestyle='--', linewidth=2, label='Input bounds'),
    plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=C_RED, markersize=10, label=r'$u_{\rm ref}$'),
    plt.Line2D([0], [0], marker='*', color='w', markerfacecolor='black', markersize=15, label=r'$u^*$'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper left')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 6. Application — Safe 2D Navigation

We deploy the CLF-CBF-QP controller for a **single-integrator robot** navigating to a goal while avoiding multiple circular obstacles.

- **Dynamics:** $\dot{x} = u$ (single integrator, $f = 0$, $g = I$)
- **CLF:** $V(x) = \|x - x_{\text{goal}}\|^2$ ensures convergence to goal
- **CBF per obstacle:** $h_i(x) = \|x - x_{\text{obs},i}\|^2 - r_i^2$ ensures collision avoidance
- **Multiple CBF constraints** are handled simultaneously in the QP

In [ ]:
# Define obstacle field
OBSTACLES = [
    {'center': np.array([2.5, 3.0]), 'radius': 0.8},
    {'center': np.array([4.5, 5.5]), 'radius': 1.0},
    {'center': np.array([5.5, 2.5]), 'radius': 0.7},
    {'center': np.array([3.5, 6.5]), 'radius': 0.6},
    {'center': np.array([6.5, 5.0]), 'radius': 0.9},
]

NAV_START = np.array([0.5, 0.5])
NAV_GOAL = X_GOAL  # (8, 8)

# Navigation CLF and CBF functions
def nav_V(x, goal=NAV_GOAL):
    """CLF for navigation: V = ||x - goal||^2.

    Args:
        x: Position. Shape: (2,).
        goal: Goal position. Shape: (2,).

    Returns:
        V: CLF value (scalar).
    """
    return np.sum((x - goal)**2)

def nav_grad_V(x, goal=NAV_GOAL):
    """Gradient of navigation CLF.

    Args:
        x: Position. Shape: (2,).
        goal: Goal position. Shape: (2,).

    Returns:
        grad_V: Gradient. Shape: (2,).
    """
    return 2.0 * (x - goal)

def make_obs_h(obs):
    """Create CBF function for a circular obstacle.

    Args:
        obs: Dict with 'center' and 'radius'.

    Returns:
        h: CBF function h(x) -> scalar.
    """
    def h(x):
        return np.sum((x - obs['center'])**2) - obs['radius']**2
    return h

def make_obs_grad_h(obs):
    """Create CBF gradient function for a circular obstacle.

    Args:
        obs: Dict with 'center' and 'radius'.

    Returns:
        grad_h: CBF gradient function (2,) -> (2,).
    """
    def grad_h(x):
        return 2.0 * (x - obs['center'])
    return grad_h

# Build CBF function lists
h_funcs = [make_obs_h(obs) for obs in OBSTACLES]
grad_h_funcs = [make_obs_grad_h(obs) for obs in OBSTACLES]

# Reference control: go-to-goal
def nav_u_ref(x, goal=NAV_GOAL):
    """Nominal go-to-goal controller.

    Args:
        x: Position. Shape: (2,).
        goal: Goal position. Shape: (2,).

    Returns:
        u_ref: Reference velocity. Shape: (2,).
    """
    d = goal - x
    norm = np.linalg.norm(d)
    if norm < 0.1:
        return np.zeros(2)
    return V_MAX * d / norm

# Single integrator dynamics
nav_f = lambda x: np.zeros(2)
nav_g = lambda x: np.eye(2)

# Run CLF-CBF-QP navigation
t_nav, x_nav, u_nav, delta_nav = simulate_clf_cbf(
    nav_f, nav_g, nav_V, nav_grad_V, h_funcs, grad_h_funcs,
    NAV_START, T=10.0, dt=DT, u_ref_func=nav_u_ref, u_dim=2,
    u_min=np.array([-V_MAX, -V_MAX]), u_max=np.array([V_MAX, V_MAX]),
    gamma=0.5, alpha=1.0, p_slack=P_SLACK
)

# Naive (unconstrained) simulation
t_naive_nav, x_naive_nav, _ = simulate(
    lambda x, u: u, lambda x, t: nav_u_ref(x), NAV_START, 10.0
)

# Verify safety
all_h_nav = np.zeros((len(t_nav), len(OBSTACLES)))
for i in range(len(t_nav)):
    for j, h_func in enumerate(h_funcs):
        all_h_nav[i, j] = h_func(x_nav[i])

min_h_all = np.min(all_h_nav)
status = "PASS" if min_h_all >= -0.1 else "FAIL"
print(f"All CBF constraints satisfied (min h = {min_h_all:.4f}): [{status}]")

# Verify convergence
final_dist = np.linalg.norm(x_nav[-1] - NAV_GOAL)
status = "PASS" if final_dist < 0.5 else "FAIL"
print(f"Robot reaches goal (final dist = {final_dist:.4f}): [{status}]")

# Check naive violates safety
all_h_naive = np.zeros((len(t_naive_nav), len(OBSTACLES)))
for i in range(len(t_naive_nav)):
    for j, h_func in enumerate(h_funcs):
        all_h_naive[i, j] = h_func(x_naive_nav[i])
min_h_naive = np.min(all_h_naive)
status = "PASS" if min_h_naive < 0 else "FAIL"
print(f"Naive controller violates safety (min h = {min_h_naive:.4f}): [{status}]")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Draw obstacles
for obs in OBSTACLES:
    circle = Circle(obs['center'], obs['radius'], color=C_UNSAFE, alpha=0.4)
    ax.add_patch(circle)
    border = Circle(obs['center'], obs['radius'], fill=False,
                    edgecolor=C_UNSAFE, linewidth=2)
    ax.add_patch(border)
    ax.plot(obs['center'][0], obs['center'][1], 'kx', markersize=6)

# Naive trajectory (dashed)
ax.plot(x_naive_nav[:, 0], x_naive_nav[:, 1], '--', color=C_RED,
        linewidth=2, alpha=0.6, label='Naive (unsafe)')

# Safe trajectory
ax.plot(x_nav[:, 0], x_nav[:, 1], '-', color=C_BLUE, linewidth=2.5,
        label='CLF-CBF-QP (safe)')

# Start and goal
ax.plot(NAV_START[0], NAV_START[1], 'go', markersize=14, label='Start', zorder=5)
ax.plot(NAV_GOAL[0], NAV_GOAL[1], 'r*', markersize=18, label='Goal', zorder=5)

# Direction arrows along safe trajectory
arrow_steps = np.arange(0, len(t_nav)-1, len(t_nav)//15)
for i in arrow_steps:
    ax.annotate('', xy=x_nav[i+5, :2], xytext=x_nav[i, :2],
                arrowprops=dict(arrowstyle='->', color=C_BLUE, lw=1.5))

ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('Safe 2D Navigation through Obstacle Field')
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim(-0.5, 9.5)
ax.set_ylim(-0.5, 9.5)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Panel 1: V(t) - CLF convergence
ax = axes[0]
V_nav = np.array([nav_V(x_nav[i]) for i in range(len(t_nav))])
ax.plot(t_nav, V_nav, color=C_BLUE, linewidth=2)
ax.set_ylabel(r'$V(x) = \|x - x_{\rm goal}\|^2$')
ax.set_title('CLF Value (Convergence to Goal)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Panel 2: All h_i(t) - CBF safety
ax = axes[1]
obs_colors = [C_BLUE, C_RED, C_GREEN, C_GOLD, C_PURPLE]
for j in range(len(OBSTACLES)):
    ax.plot(t_nav, all_h_nav[:, j], color=obs_colors[j], linewidth=1.5,
            label=f'$h_{j+1}$ (obs {j+1})')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.7, label='Safety boundary')
ax.fill_between(t_nav, -5, 0, color=C_UNSAFE, alpha=0.1)
ax.set_ylabel(r'$h_i(x)$')
ax.set_title('CBF Values (Must Stay $\\geq 0$)')
ax.legend(fontsize=9, ncol=3, loc='upper right')
ax.set_ylim(bottom=-2)

# Panel 3: ||u||(t) - Control effort
ax = axes[2]
u_norms = np.linalg.norm(u_nav, axis=1)
ax.plot(t_nav[:-1], u_norms, color=C_GREEN, linewidth=2)
ax.axhline(y=V_MAX, color=C_RED, linestyle='--', linewidth=1, label=f'$V_{{max}} = {V_MAX}$')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\|u\|$')
ax.set_title('Control Effort')
ax.legend()

plt.tight_layout()
plt.show()

---
## 7. Application — Adaptive Cruise Control

### Problem Setup

A classic safety-critical control problem: an ego vehicle must maintain desired speed while keeping a safe following distance from a lead vehicle.

**Ego vehicle dynamics** (double integrator):
- State: $x = [p_{\text{ego}}, v_{\text{ego}}]$ (position, velocity)
- Control: $u = a$ (acceleration), bounded by $a_{\min} \leq u \leq a_{\max}$
- Dynamics: $\dot{p} = v$, $\dot{v} = u$

**Safety (CBF):** Time-headway criterion:
$$h(x) = p_{\text{lead}}(t) - p_{\text{ego}} - \tau_h \cdot v_{\text{ego}}$$
where $\tau_h$ is the desired time headway. $h \geq 0$ means safe following distance.

**Stability (CLF):** Speed regulation:
$$V(x) = (v_{\text{ego}} - v_{\text{des}})^2$$

**Scenario:** Lead vehicle cruises at 30 m/s, then brakes hard at $t = 5$ s.

In [ ]:
# Lead vehicle trajectory
def lead_vehicle(t):
    """Lead vehicle position and velocity.

    Cruises at V_DES, then brakes hard at t=5s.

    Args:
        t: Time (scalar).

    Returns:
        p_lead: Position (scalar).
        v_lead: Velocity (scalar).
    """
    t_brake = 5.0
    a_brake = -4.0  # Lead braking deceleration

    if t < t_brake:
        p = 100.0 + V_DES * t
        v = V_DES
    else:
        dt_b = t - t_brake
        v = max(V_DES + a_brake * dt_b, 10.0)  # Don't go below 10 m/s
        # Position: integral of velocity
        if V_DES + a_brake * dt_b >= 10.0:
            p = 100.0 + V_DES * t_brake + V_DES * dt_b + 0.5 * a_brake * dt_b**2
        else:
            # Time when v reaches 10
            t_10 = (10.0 - V_DES) / a_brake
            p_10 = 100.0 + V_DES * t_brake + V_DES * t_10 + 0.5 * a_brake * t_10**2
            p = p_10 + 10.0 * (dt_b - t_10)
    return p, v


def acc_clf_cbf_controller(x, t):
    """ACC controller using CLF-CBF-QP.

    State x = [p_ego, v_ego], control u = acceleration.

    Args:
        x: Ego state [position, velocity]. Shape: (2,).
        t: Current time (scalar).

    Returns:
        u: Acceleration command (scalar as array). Shape: (1,).
    """
    p_ego, v_ego = x[0], x[1]
    p_lead, v_lead = lead_vehicle(t)

    # Dynamics: f(x) = [v, 0], g(x) = [0, 1]
    f_x = np.array([v_ego, 0.0])
    g_x = np.array([[0.0], [1.0]])

    # CLF: V = (v - v_des)^2
    V_val = (v_ego - V_DES)**2
    grad_V = np.array([0.0, 2.0 * (v_ego - V_DES)])

    # CBF: h = p_lead - p_ego - tau*v_ego
    h_val = p_lead - p_ego - TAU_H * v_ego
    # grad_h w.r.t. x = [-1, -tau]
    # But h also depends on time through p_lead — we handle this:
    # dh/dt = v_lead - v_ego - tau*u  (using chain rule)
    # For the QP: Lfh = grad_h @ f = -v_ego + (-tau)*0 = ... wait
    # More carefully: h(x,t) = p_lead(t) - p_ego - tau*v_ego
    # dh/dt = dp_lead/dt - dp_ego/dt - tau*dv_ego/dt
    #       = v_lead - v_ego - tau*u
    # So: Lfh = v_lead - v_ego, Lgh = -tau
    # We encode this as: grad_h = [-1, -tau], but add v_lead to Lfh
    grad_h = np.array([-1.0, -TAU_H])
    # Lfh = grad_h @ f_x = -v_ego  (but we need + v_lead)
    # We'll adjust: define effective Lfh including v_lead
    Lfh_eff = v_lead - v_ego
    Lgh_eff = -TAU_H

    # Reference: accelerate toward desired speed
    u_ref = np.array([2.0 * (V_DES - v_ego)])

    # QP: min (u - u_ref)^2 + p*delta^2
    # CLF: grad_V @ f + grad_V @ g * u <= -gamma*V + delta
    #       => 0 + 2(v-v_des)*u <= -gamma*(v-v_des)^2 + delta
    # CBF: Lfh_eff + Lgh_eff * u >= -alpha * h
    #       => (v_lead - v_ego) - tau*u >= -alpha * h

    LfV = grad_V @ f_x   # = 0
    LgV = grad_V @ g_x   # = 2*(v-v_des)

    def objective(z):
        u = z[0]
        delta = z[1]
        return (u - u_ref[0])**2 + P_SLACK * delta**2

    def clf_con(z):
        u, delta = z[0], z[1]
        # -(LfV + LgV*u + gamma*V - delta) >= 0
        return -(LfV + LgV.flatten()[0] * u + GAMMA_CLF * V_val - delta)

    def cbf_con(z):
        u = z[0]
        # Lfh_eff + Lgh_eff*u + alpha*h >= 0
        return Lfh_eff + Lgh_eff * u + ALPHA_CBF * h_val

    constraints = [
        {'type': 'ineq', 'fun': clf_con},
        {'type': 'ineq', 'fun': cbf_con},
    ]
    bounds = [(A_MIN, A_MAX), (None, None)]

    z0 = np.array([u_ref[0], 0.0])
    result = minimize(objective, z0, method='SLSQP',
                      constraints=constraints, bounds=bounds,
                      options={'ftol': 1e-10})

    return np.array([result.x[0]])


# Simulate ACC
acc_x0 = np.array([0.0, V_DES])  # Start at origin, at desired speed
acc_dynamics = lambda x, u: np.array([x[1], u[0]])

t_acc, x_acc, u_acc = simulate(acc_dynamics, acc_clf_cbf_controller, acc_x0, 15.0, dt=0.02)

# Compute lead vehicle trajectory
p_lead_hist = np.array([lead_vehicle(t)[0] for t in t_acc])
v_lead_hist = np.array([lead_vehicle(t)[1] for t in t_acc])

# Safety metric: h(t)
h_acc = p_lead_hist - x_acc[:, 0] - TAU_H * x_acc[:, 1]

# Verify safety
min_h_acc = np.min(h_acc)
status = "PASS" if min_h_acc >= -0.5 else "FAIL"
print(f"ACC safety maintained (min h = {min_h_acc:.4f}): [{status}]")

# Verify ego doesn't exceed speed limit significantly
max_v = np.max(x_acc[:, 1])
status = "PASS" if max_v <= V_DES + 1.0 else "FAIL"
print(f"Ego speed bounded (max v = {max_v:.2f} m/s): [{status}]")

When the lead vehicle brakes at $t = 5$ s, the CBF constraint activates and forces the ego vehicle to decelerate — maintaining the time-headway safety distance. The CLF slack variable allows the speed regulation objective to be temporarily violated for safety.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Positions
ax = axes[0, 0]
ax.plot(t_acc, x_acc[:, 0], color=C_BLUE, linewidth=2, label='Ego')
ax.plot(t_acc, p_lead_hist, color=C_RED, linewidth=2, label='Lead')
ax.axvline(x=5.0, color='gray', linestyle='--', alpha=0.5, label='Lead brakes')
ax.set_ylabel('Position (m)')
ax.set_title('(a) Vehicle Positions')
ax.legend()

# (b) Velocities
ax = axes[0, 1]
ax.plot(t_acc, x_acc[:, 1], color=C_BLUE, linewidth=2, label='Ego')
ax.plot(t_acc, v_lead_hist, color=C_RED, linewidth=2, label='Lead')
ax.axhline(y=V_DES, color=C_GOLD, linestyle='--', alpha=0.7, label=f'$v_{{des}} = {V_DES}$')
ax.axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Velocity (m/s)')
ax.set_title('(b) Vehicle Velocities')
ax.legend()

# (c) Inter-vehicle distance with safety threshold
ax = axes[1, 0]
gap = p_lead_hist - x_acc[:, 0]
safe_dist = TAU_H * x_acc[:, 1]
ax.plot(t_acc, gap, color=C_BLUE, linewidth=2, label='Actual gap')
ax.plot(t_acc, safe_dist, color=C_RED, linewidth=2, linestyle='--',
        label=f'Safety threshold ($\\tau v$)')
ax.fill_between(t_acc, 0, safe_dist, color=C_UNSAFE, alpha=0.1)
ax.axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Distance (m)')
ax.set_title('(c) Following Distance vs. Safety Threshold')
ax.legend()

# (d) Acceleration command
ax = axes[1, 1]
ax.plot(t_acc[:-1], u_acc[:, 0], color=C_GREEN, linewidth=2, label='$u$ (accel)')
ax.axhline(y=A_MAX, color=C_RED, linestyle='--', alpha=0.5, label=f'$a_{{max}} = {A_MAX}$')
ax.axhline(y=A_MIN, color=C_RED, linestyle='--', alpha=0.5, label=f'$a_{{min}} = {A_MIN}$')
ax.fill_between(t_acc[:-1], A_MIN, A_MAX, color=C_GREEN, alpha=0.05)
ax.axvline(x=5.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Acceleration (m/s²)')
ax.set_title('(d) Acceleration Command')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 8. Higher-Relative-Degree CBFs (HOCBFs)

### The Problem

When the barrier function $h$ has **relative degree** greater than 1 (i.e., $L_g h = 0$), the standard CBF condition $L_f h + L_g h \cdot u \geq -\alpha(h)$ becomes independent of $u$ — the control input doesn't appear. This happens when the constraint involves position but the control acts on acceleration (e.g., double integrator with position constraint).

### HOCBF for Relative Degree 2

For a system where $h$ has relative degree 2, define:

$$\psi_0(x) = h(x)$$
$$\psi_1(x) = \dot{\psi}_0 + \alpha_1(\psi_0) = L_f h + \alpha_1(h)$$

The **HOCBF constraint** is:

$$\dot{\psi}_1 + \alpha_2(\psi_1) \geq 0$$

Expanding:

$$L_f^2 h + L_g L_f h \cdot u + \alpha_1'(h) \cdot L_f h + \alpha_2(\psi_1) \geq 0$$

Now the control $u$ appears (since $L_g L_f h \neq 0$). Using linear class-K functions $\alpha_i(r) = \gamma_i r$:

$$L_f^2 h + L_g L_f h \cdot u + \gamma_1 L_f h + \gamma_2 (L_f h + \gamma_1 h) \geq 0$$

In [ ]:
def hocbf_constraint(x, f_x, g_x, h_val, Lfh, Lf2h, LgLfh, gamma1=2.0, gamma2=2.0):
    """Compute HOCBF constraint value for relative degree 2.

    The constraint is: Lf2h + LgLfh*u + gamma1*Lfh + gamma2*(Lfh + gamma1*h) >= 0
    Returns coefficients (a, b) such that constraint is: a + b*u >= 0

    Args:
        x: State vector. Shape: (n,).
        f_x: Drift dynamics. Shape: (n,).
        g_x: Input matrix. Shape: (n, m).
        h_val: Barrier function value (scalar).
        Lfh: L_f h (scalar).
        Lf2h: L_f^2 h (scalar).
        LgLfh: L_g L_f h. Shape: (m,).
        gamma1: First class-K coefficient (scalar).
        gamma2: Second class-K coefficient (scalar).

    Returns:
        a_coeff: Constant term (scalar).
        b_coeff: Control coefficient. Shape: (m,).
    """
    psi1 = Lfh + gamma1 * h_val
    a_coeff = Lf2h + gamma1 * Lfh + gamma2 * psi1
    b_coeff = LgLfh
    return a_coeff, b_coeff


# ---- Teaching example: Double integrator with position constraint ----
# State: x = [position, velocity], control: u = acceleration
# Constraint: x1 >= -1, i.e., h = x1 + 1
# Relative degree 2: L_g h = [0, 1] @ [0; 1] -- wait:
# h = x1 + 1 => grad_h = [1, 0]
# L_f h = grad_h @ f = [1, 0] @ [x2, 0] = x2
# L_g h = grad_h @ g = [1, 0] @ [0; 1] = 0  -- relative degree > 1!
# L_f^2 h = d(Lfh)/dx @ f = [0, 1] @ [x2, 0] = 0
# L_g L_f h = d(Lfh)/dx @ g = [0, 1] @ [0; 1] = 1  -- control appears!

def di_dynamics(x, u):
    """Double integrator dynamics.

    Args:
        x: State [position, velocity]. Shape: (2,).
        u: Acceleration. Shape: (1,).

    Returns:
        dx: State derivative. Shape: (2,).
    """
    return np.array([x[1], u[0]])

def di_naive_cbf_controller(x, t, gamma=2.0):
    """Naive (incorrect) CBF controller for double integrator.

    Tries to use standard CBF (ignoring relative degree issue).

    Args:
        x: State [position, velocity]. Shape: (2,).
        t: Time (unused).
        gamma: Class-K coefficient.

    Returns:
        u: Acceleration command. Shape: (1,).
    """
    h_val = x[0] + 1.0
    Lfh = x[1]
    Lgh = 0.0  # This is zero! Standard CBF doesn't work

    # Nominal: drive to origin
    u_nom = -2.0 * x[0] - 3.0 * x[1]

    # Standard CBF would try: Lfh + Lgh*u >= -gamma*h
    # But Lgh = 0, so constraint is: x2 >= -gamma*(x1+1)
    # This is a state constraint, not a control constraint!
    # Just return nominal — safety is NOT guaranteed
    return np.array([u_nom])

def di_hocbf_controller(x, t, gamma1=2.0, gamma2=2.0):
    """HOCBF controller for double integrator with position constraint.

    Args:
        x: State [position, velocity]. Shape: (2,).
        t: Time (unused).
        gamma1: First HOCBF parameter.
        gamma2: Second HOCBF parameter.

    Returns:
        u: Acceleration command. Shape: (1,).
    """
    h_val = x[0] + 1.0     # h = x1 + 1
    Lfh = x[1]              # L_f h = x2
    Lf2h = 0.0              # L_f^2 h = 0
    LgLfh = np.array([1.0]) # L_g L_f h = 1

    a_coeff, b_coeff = hocbf_constraint(x, np.array([x[1], 0.0]),
                                         np.array([[0.0], [1.0]]),
                                         h_val, Lfh, Lf2h, LgLfh,
                                         gamma1, gamma2)

    # Nominal control: stabilize to origin
    u_nom = -2.0 * x[0] - 3.0 * x[1]

    # QP: min (u - u_nom)^2 s.t. a + b*u >= 0
    def objective(u):
        return (u[0] - u_nom)**2

    def hocbf_con(u):
        return a_coeff + b_coeff[0] * u[0]

    constraints = [{'type': 'ineq', 'fun': hocbf_con}]
    bounds = [(-10.0, 10.0)]

    result = minimize(objective, np.array([u_nom]), method='SLSQP',
                      constraints=constraints, bounds=bounds)
    return np.array([result.x[0]])


# Simulate: start at x=[0, -3] — heading toward constraint boundary fast
hocbf_x0 = np.array([0.0, -3.0])

t_naive_di, x_naive_di, _ = simulate(di_dynamics, di_naive_cbf_controller, hocbf_x0, 6.0)
t_hocbf, x_hocbf, _ = simulate(di_dynamics, di_hocbf_controller, hocbf_x0, 6.0)

# Verify
min_pos_naive = np.min(x_naive_di[:, 0])
min_pos_hocbf = np.min(x_hocbf[:, 0])

status = "PASS" if min_pos_naive < -1.0 else "FAIL"
print(f"Naive CBF violates x1 >= -1 constraint (min x1 = {min_pos_naive:.4f}): [{status}]")

status = "PASS" if min_pos_hocbf >= -1.05 else "FAIL"
print(f"HOCBF respects x1 >= -1 constraint (min x1 = {min_pos_hocbf:.4f}): [{status}]")

The naive CBF attempt fails because the standard CBF condition has no control authority ($L_g h = 0$). The HOCBF correctly identifies the second-order structure and produces a constraint where the control input appears, successfully keeping $x_1 \geq -1$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Position trajectories
ax = axes[0]
ax.plot(t_naive_di, x_naive_di[:, 0], '--', color=C_RED, linewidth=2,
        label='Naive CBF (fails)')
ax.plot(t_hocbf, x_hocbf[:, 0], '-', color=C_BLUE, linewidth=2.5,
        label='HOCBF (succeeds)')
ax.axhline(y=-1.0, color='black', linewidth=2, linestyle='-', label='Constraint: $x_1 = -1$')
ax.fill_between(t_naive_di, -3, -1, color=C_UNSAFE, alpha=0.15, label='Unsafe region')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'Position $x_1$')
ax.set_title('Position Constraint: Naive CBF vs. HOCBF')
ax.legend()
ax.set_ylim(-2.5, 1.0)

# Right: Phase portrait
ax = axes[1]
ax.plot(x_naive_di[:, 0], x_naive_di[:, 1], '--', color=C_RED, linewidth=2,
        label='Naive CBF')
ax.plot(x_hocbf[:, 0], x_hocbf[:, 1], '-', color=C_BLUE, linewidth=2.5,
        label='HOCBF')

# Constraint boundary
ax.axvline(x=-1.0, color='black', linewidth=2, linestyle='-')
ax.fill_betweenx(np.linspace(-4, 4, 10), -3, -1, color=C_UNSAFE, alpha=0.15)

# Start point
ax.plot(hocbf_x0[0], hocbf_x0[1], 'go', markersize=12, label='Start', zorder=5)
ax.plot(0, 0, 'k*', markersize=15, label='Origin (target)', zorder=5)

ax.set_xlabel(r'Position $x_1$')
ax.set_ylabel(r'Velocity $x_2$')
ax.set_title('Phase Portrait: HOCBF Keeps System Safe')
ax.legend()
ax.set_xlim(-2.5, 1.5)
ax.set_ylim(-4, 4)

plt.tight_layout()
plt.show()

---
## 9. Summary

We have built a complete pipeline from stability theory to safe control:

1. **Lyapunov stability** — certify stability without solving ODEs using energy-like functions
2. **Control Lyapunov Functions** — design stabilizing controllers for control-affine systems
3. **Control Barrier Functions** — enforce safety (forward invariance) through linear constraints on control
4. **CLF-CBF-QP** — unify stability + safety in a single QP, with safety taking strict priority
5. **Applications** — 2D obstacle avoidance navigation and adaptive cruise control
6. **HOCBFs** — extend barrier functions to higher-relative-degree systems

### Extensions
- **Robust CBFs:** handle model uncertainty and disturbances
- **Multi-agent CBFs:** distributed safe control for robot swarms
- **Learning-based CBFs:** learn barrier functions from data (neural CBFs)
- **Hamilton-Jacobi reachability:** compute exact safe sets via HJ PDEs
- **Stochastic CBFs:** safety under stochastic dynamics and measurement noise

In [ ]:
# ---- 2x2 Summary Figure ----
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# (a) Lyapunov level sets + phase portrait
ax = axes[0, 0]
grid_s = np.linspace(-1.0, 1.0, 30)
X1s, X2s = np.meshgrid(grid_s, grid_s)
Vs = X1s**2 + X2s**2
ax.contour(X1s, X2s, Vs, levels=[0.1, 0.3, 0.5, 0.8], colors='gray',
           linestyles='--', linewidths=0.8)
U_f = np.zeros_like(X1s)
V_f = np.zeros_like(X1s)
for i in range(X1s.shape[0]):
    for j in range(X1s.shape[1]):
        f = lyapunov_example_dynamics(np.array([X1s[i, j], X2s[i, j]]))
        U_f[i, j] = f[0]
        V_f[i, j] = f[1]
ax.streamplot(grid_s, grid_s, U_f, V_f, color='steelblue', density=1.2,
              linewidth=0.8, arrowsize=1)
ax.plot(0, 0, 'k*', markersize=12)
ax.set_title('(a) Lyapunov Stability Analysis')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_aspect('equal')

# (b) CBF safe set with trajectories
ax = axes[0, 1]
theta = np.linspace(0, 2*np.pi, 200)
ax.fill(OBS_CENTER[0] + OBS_RADIUS*np.cos(theta),
        OBS_CENTER[1] + OBS_RADIUS*np.sin(theta),
        color=C_UNSAFE, alpha=0.3)
ax.plot(OBS_CENTER[0] + OBS_RADIUS*np.cos(theta),
        OBS_CENTER[1] + OBS_RADIUS*np.sin(theta),
        color=C_UNSAFE, linewidth=2)
ax.plot(x_safe[:, 0], x_safe[:, 1], '-', color=C_BLUE, linewidth=2)
ax.plot(x_naive[:, 0], x_naive[:, 1], '--', color=C_RED, linewidth=1.5, alpha=0.6)
ax.plot(CBF_START[0], CBF_START[1], 'go', markersize=10)
ax.plot(CBF_TARGET[0], CBF_TARGET[1], 'r*', markersize=12)
ax.set_title('(b) CBF Safe Navigation')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_xlim(-0.5, 8.5)
ax.set_ylim(-0.5, 6.5)
ax.set_aspect('equal')

# (c) 2D navigation through obstacles
ax = axes[1, 0]
for obs in OBSTACLES:
    circle = Circle(obs['center'], obs['radius'], color=C_UNSAFE, alpha=0.4)
    ax.add_patch(circle)
    border = Circle(obs['center'], obs['radius'], fill=False,
                    edgecolor=C_UNSAFE, linewidth=1.5)
    ax.add_patch(border)
ax.plot(x_nav[:, 0], x_nav[:, 1], '-', color=C_BLUE, linewidth=2.5)
ax.plot(NAV_START[0], NAV_START[1], 'go', markersize=10)
ax.plot(NAV_GOAL[0], NAV_GOAL[1], 'r*', markersize=14)
ax.set_title('(c) Multi-Obstacle Safe Navigation')
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_xlim(-0.5, 9.5)
ax.set_ylim(-0.5, 9.5)
ax.set_aspect('equal')

# (d) ACC braking scenario
ax = axes[1, 1]
ax.plot(t_acc, x_acc[:, 1], color=C_BLUE, linewidth=2, label='Ego velocity')
ax.plot(t_acc, v_lead_hist, color=C_RED, linewidth=2, label='Lead velocity')
ax.axhline(y=V_DES, color=C_GOLD, linestyle='--', alpha=0.7, label=r'$v_{des}$')
ax.axvline(x=5.0, color='gray', linestyle='--', alpha=0.5, label='Lead brakes')
ax.set_title('(d) Adaptive Cruise Control')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Velocity (m/s)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\nNotebook complete. All CLF-CBF-QP concepts demonstrated with working code.")